# Subasa Sinhala OCR — Benchmark Evaluation

Benchmark evaluation of the **[Subasa OCR](https://ocr.subasa.lk/)** web service on the `avishadilhara/sinhala-ocr-lk-acts-1010` test set (202 images).

**Metrics:** CER, WER, BLEU, ANLS, METEOR, Character Accuracy

**Method:** Headless Selenium uploads each test image to the Subasa OCR website and extracts the OCR text.

> **Kaggle Settings Required:** Internet must be **ON** (Settings → Internet → On)

> **Fresh Run:** Delete any old `checkpoint.pkl` before running to ensure clean results.

## 1. Install Dependencies

In [ ]:
%%bash
# Install Google Chrome for headless Selenium
apt-get update -qq
wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
apt-get install -y -qq ./google-chrome-stable_current_amd64.deb > /dev/null 2>&1
rm google-chrome-stable_current_amd64.deb
google-chrome --version

In [ ]:
%%capture
!pip install -q selenium webdriver-manager
!pip install -q datasets huggingface-hub>=1.3.1
!pip install -q jiwer
!pip install -q python-Levenshtein
!pip install -q Pillow>=11.3.0

## 2. Imports & Configuration

In [ ]:
import os
import re
import json
import time
import glob
import pickle
import tempfile
from pathlib import Path
from tqdm import tqdm
from collections import defaultdict
import pandas as pd
from PIL import Image

# Selenium
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from webdriver_manager.chrome import ChromeDriverManager

# Metrics
from jiwer import cer, wer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
import nltk

import warnings
warnings.filterwarnings('ignore')

try:
    nltk.data.find('wordnet')
except LookupError:
    nltk.download('wordnet', quiet=True)
    nltk.download('punkt', quiet=True)
    nltk.download('omw-1.4', quiet=True)

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

OCR_URL = "https://ocr.subasa.lk/"

# Max seconds to wait for OCR processing per image
OCR_TIMEOUT = 180

# Save checkpoint every N samples
CHECKPOINT_INTERVAL = 5

# Max retries per image if upload/extraction fails
MAX_RETRIES = 3

# Output directory
OUTPUT_DIR = "./test_results"
INFERENCE_DIR = os.path.join(OUTPUT_DIR, "inference_outputs")
TEMP_IMG_DIR = os.path.join(OUTPUT_DIR, "temp_images")

os.makedirs(INFERENCE_DIR, exist_ok=True)
os.makedirs(os.path.join(INFERENCE_DIR, "images"), exist_ok=True)
os.makedirs(TEMP_IMG_DIR, exist_ok=True)

print("Configuration loaded.")

## 3. Load Test Dataset

In [ ]:
from datasets import load_dataset
from huggingface_hub import login

# Login to Hugging Face (if private dataset)
login()  # Enter your token

In [ ]:
# ========================================
# LOAD TEST DATASET
# ========================================
print("\n" + "="*80)
print(" LOADING TEST DATASET")
print("="*80)

print("\nLoading dataset from Hugging Face...")
dataset = load_dataset("avishadilhara/sinhala-ocr-lk-acts-1010")

print(f" Dataset loaded!")
print(f"   Test samples: {len(dataset['test'])}")

## 4. Boilerplate Removal & Selenium Helpers

In [ ]:
# ============================================================
# BOILERPLATE REMOVAL
# The Subasa OCR website has static footer content that gets
# scraped alongside the OCR output. We MUST strip it or every
# metric (CER, WER, BLEU, ANLS, METEOR) will be wrong.
# ============================================================

BOILERPLATE_CUTOFF_PHRASES = [
    "Isuri Nanomi",
    "Lakshmi Madhushika",
    "Chamila Liyanage",
    "Contributors",
    "Acknowledgement",
    "Active contribution of Theekshana",
    "Funded by",
    "Information and Communication Technology Agency",
    "PAN Localization",
    "Disclaimer",
    "Every effort is made to provide",
    "How to cite",
    "@Misc{tts",
    "LTRL UCSC",
    "author = {",
    "title = {Sinhala OCR}",
    "url = {https://ocr.subasa",
    "version ={1.0.0}",
    "privately owned rights",
]

UI_LABELS_TO_REMOVE = [
    "Sinhala OCR",
    "Select Image",
    "Uploaded Image",
    "Extracted Text",
    "Home",
    "Powered By",
    "Browse files",
    "Drag and drop file here",
    "Limit 200MB per file",
    "JPG, PNG, JPEG",
    "Contributors",
    "Acknowledgement",
    "Funded by",
    "Disclaimer",
    "How to cite",
    "PAN Localization",
]


def strip_boilerplate(text: str) -> str:
    """Remove ALL Subasa website boilerplate and UI labels from text."""
    if not text:
        return ""

    lines = text.split("\n")
    truncated_lines = []
    for line in lines:
        is_boilerplate = False
        for phrase in BOILERPLATE_CUTOFF_PHRASES:
            if phrase in line:
                is_boilerplate = True
                break
        if is_boilerplate:
            break
        truncated_lines.append(line)

    cleaned_lines = []
    for line in truncated_lines:
        stripped = line.strip()
        if not stripped:
            continue
        if any(stripped == lbl or stripped.startswith(lbl) for lbl in UI_LABELS_TO_REMOVE):
            continue
        cleaned_lines.append(stripped)

    return "\n".join(cleaned_lines).strip()


_test = """Sinhala OCR
Select Image
Uploaded Image
Extracted Text
2002 අංක 1 දරන පළාත පාලන
සෑම තැනැත්තකු ම
Contributors
Isuri Nanomi Arachchige, Lakshmi Madhushika
Disclaimer
How to cite"""

_result = strip_boilerplate(_test)
assert "Contributors" not in _result, f"Failed! Got: {_result}"
assert "Isuri" not in _result, f"Failed! Got: {_result}"
assert "Sinhala OCR" not in _result, f"Failed! Got: {_result}"
assert "2002 අංක" in _result, f"Failed! Got: {_result}"
print(f" strip_boilerplate() verified. Sample output:\n  {_result[:80]}...")

In [ ]:
def create_driver() -> webdriver.Chrome:
    """Create a headless Chrome WebDriver for Kaggle."""
    chrome_options = Options()
    chrome_options.add_argument("--headless=new")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--window-size=1920,1080")
    chrome_options.add_argument("--log-level=3")
    chrome_options.add_argument("--disable-extensions")
    chrome_options.add_argument("--disable-infobars")
    chrome_options.add_experimental_option("excludeSwitches", ["enable-logging"])

    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=chrome_options)
    return driver


def upload_and_extract(driver: webdriver.Chrome, image_path: str, timeout: int = 180) -> str:
    """
    Upload an image to Subasa OCR and extract the result text.
    Returns OCR text with all website boilerplate stripped.
    """
    # Navigate
    driver.get(OCR_URL)
    WebDriverWait(driver, 30).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, 'input[type="file"]'))
    )
    time.sleep(2)

    # Upload
    file_input = driver.find_element(By.CSS_SELECTOR, 'input[type="file"]')
    file_input.send_keys(os.path.abspath(image_path))

    # Wait for Streamlit spinner
    spinner_css = '[data-testid="stStatusWidget"], .stSpinner, [data-testid="stSpinner"]'
    try:
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, spinner_css))
        )
    except TimeoutException:
        pass
    try:
        WebDriverWait(driver, timeout).until_not(
            EC.presence_of_element_located((By.CSS_SELECTOR, spinner_css))
        )
    except TimeoutException:
        pass

    time.sleep(5)

    # ---- Extract raw text ----
    raw_text = ""

    # Try Streamlit selectors first
    for sel in ['[data-testid="stText"]', '[data-testid="stMarkdown"]',
                '[data-testid="stCode"]', 'textarea', 'pre']:
        try:
            for el in driver.find_elements(By.CSS_SELECTOR, sel):
                text = (el.text or el.get_attribute("value") or "").strip()
                if text and len(text) > 5:
                    raw_text += text + "\n"
        except NoSuchElementException:
            continue

    # Fallback: full body text
    if not raw_text:
        try:
            raw_text = driver.find_element(By.TAG_NAME, "body").text
        except Exception:
            pass

    # ALWAYS strip boilerplate before returning
    return strip_boilerplate(raw_text)


print(" Selenium helper functions defined.")

## 5. Metrics Functions

In [ ]:
def calculate_anls(ground_truth, prediction, threshold=0.5):
    """Calculate Average Normalized Levenshtein Similarity (ANLS)"""
    try:
        from Levenshtein import distance as levenshtein_distance
        if len(ground_truth) == 0 and len(prediction) == 0:
            return 1.0
        if len(ground_truth) == 0 or len(prediction) == 0:
            return 0.0
        edit_distance = levenshtein_distance(ground_truth, prediction)
        max_len = max(len(ground_truth), len(prediction))
        normalized_distance = edit_distance / max_len
        return 1 - normalized_distance if normalized_distance < threshold else 0.0
    except ImportError:
        from jiwer import cer
        return 1 - cer(ground_truth, prediction)


def calculate_all_metrics(ground_truth, prediction):
    """Calculate all metrics: CER, WER, BLEU, ANLS, METEOR"""
    metrics = {}

    try:
        metrics['CER'] = min(cer(ground_truth, prediction), 1.0)
    except:
        metrics['CER'] = 1.0

    try:
        metrics['WER'] = min(wer(ground_truth, prediction), 1.0)
    except:
        metrics['WER'] = 1.0

    try:
        smoothing = SmoothingFunction().method1
        metrics['BLEU'] = max(0.0, min(sentence_bleu(
            [list(ground_truth)], list(prediction), smoothing_function=smoothing), 1.0))
    except:
        metrics['BLEU'] = 0.0

    try:
        metrics['ANLS'] = max(0.0, min(calculate_anls(ground_truth, prediction), 1.0))
    except:
        metrics['ANLS'] = 0.0

    try:
        metrics['METEOR'] = max(0.0, min(
            meteor_score([list(ground_truth)], list(prediction)), 1.0))
    except:
        metrics['METEOR'] = 0.0

    metrics['Char_Accuracy'] = max((1 - metrics['CER']) * 100, 0.0)
    return metrics


print(" Metrics functions defined.")

## 6. Checkpoint Management

In [ ]:
def save_checkpoint(all_results, year_results, last_processed_idx, output_dir):
    checkpoint_path = os.path.join(output_dir, "checkpoint.pkl")
    with open(checkpoint_path, 'wb') as f:
        pickle.dump({'all_results': all_results, 'year_results': dict(year_results),
                     'last_processed_idx': last_processed_idx}, f)

def load_checkpoint(output_dir):
    checkpoint_path = os.path.join(output_dir, "checkpoint.pkl")
    if os.path.exists(checkpoint_path):
        with open(checkpoint_path, 'rb') as f:
            return pickle.load(f)
    return None

print(" Checkpoint functions defined.")

## 7. Run Benchmark

In [ ]:
def run_single_inference(driver, image, idx):
    """Run OCR on a single image via Subasa OCR website."""
    if image.mode == 'RGBA':
        rgb = Image.new('RGB', image.size, (255, 255, 255))
        rgb.paste(image, mask=image.split()[3])
        image = rgb
    elif image.mode != 'RGB':
        image = image.convert('RGB')

    temp_path = os.path.join(TEMP_IMG_DIR, f"temp_{idx:04d}.png")
    image.save(temp_path, format='PNG')

    ref_path = os.path.join(INFERENCE_DIR, "images", f"{idx:04d}.png")
    image.save(ref_path, format='PNG')

    prediction = ""
    for attempt in range(MAX_RETRIES):
        try:
            prediction = upload_and_extract(driver, temp_path, timeout=OCR_TIMEOUT)
            if prediction:
                break
        except Exception as e:
            if attempt < MAX_RETRIES - 1:
                time.sleep(5)
            else:
                raise e

    try:
        os.remove(temp_path)
    except:
        pass

    prediction = strip_boilerplate(prediction)

    return prediction


def test_entire_dataset(resume=True):
    """Test Subasa OCR on entire test dataset with checkpoint support."""
    all_results = []
    year_results = defaultdict(list)
    start_idx = 0

    if resume:
        checkpoint = load_checkpoint(OUTPUT_DIR)
        if checkpoint:
            all_results = checkpoint['all_results']
            year_results = defaultdict(list, checkpoint['year_results'])
            start_idx = checkpoint['last_processed_idx'] + 1
            print("=" * 80)
            print("CHECKPOINT FOUND - RESUMING FROM PREVIOUS RUN")
            print("=" * 80)
            print(f"Already processed: {len(all_results)} samples")
            print(f"Resuming from sample: {start_idx}")
            print("=" * 80)

    test_dataset = dataset['test']
    total_samples = len(test_dataset)

    if start_idx >= total_samples:
        print("All samples already processed!")
        return all_results, year_results

    if start_idx == 0:
        print("=" * 80)
        print("BENCHMARKING SUBASA OCR ON TEST DATASET")
        print("=" * 80)
        print(f"Total samples: {total_samples} | Checkpoint every {CHECKPOINT_INTERVAL}")
        print(f"Max retries: {MAX_RETRIES} | Timeout: {OCR_TIMEOUT}s")
        print("=" * 80)

    print("\n Starting headless Chrome...")
    driver = create_driver()
    print(" Browser ready.\n")

    consecutive_errors = 0
    MAX_CONSECUTIVE_ERRORS = 5

    try:
        with tqdm(total=total_samples, initial=start_idx, desc="Testing",
                  unit="sample", ncols=100) as pbar:
            for idx in range(start_idx, total_samples):
                try:
                    sample = test_dataset[idx]
                    ground_truth = sample['text']
                    year = sample.get('year', 'Unknown')
                    image = sample['image']

                    prediction = run_single_inference(driver, image, idx)

                    pred_file = os.path.join(INFERENCE_DIR, f"result_{idx:04d}.txt")
                    with open(pred_file, 'w', encoding='utf-8') as f:
                        f.write(prediction)

                    if prediction:
                        metrics = calculate_all_metrics(ground_truth, prediction)
                    else:
                        metrics = {'CER': 1.0, 'WER': 1.0, 'BLEU': 0.0,
                                   'ANLS': 0.0, 'METEOR': 0.0, 'Char_Accuracy': 0.0}

                    result = {'sample_idx': idx, 'year': year,
                              'ground_truth_length': len(ground_truth),
                              'prediction_length': len(prediction), **metrics}
                    all_results.append(result)
                    year_results[year].append(result)
                    consecutive_errors = 0

                    pbar.set_postfix({'CER': f"{metrics['CER']:.3f}",
                                     'Acc': f"{metrics['Char_Accuracy']:.1f}"})
                    pbar.update(1)

                    if (idx + 1) % CHECKPOINT_INTERVAL == 0:
                        save_checkpoint(all_results, year_results, idx, OUTPUT_DIR)
                        pbar.write(f" Checkpoint saved at sample {idx + 1}")

                except KeyboardInterrupt:
                    print("\n Interrupted! Saving checkpoint...")
                    save_checkpoint(all_results, year_results, idx - 1, OUTPUT_DIR)
                    print(f"Processed {len(all_results)}/{total_samples}. Resume by re-running.")
                    raise

                except Exception as e:
                    consecutive_errors += 1
                    pbar.write(f"Error at sample {idx}: {str(e)[:80]}")
                    all_results.append({'sample_idx': idx,
                        'year': year if 'year' in dir() else 'Unknown',
                        'CER': 1.0, 'WER': 1.0, 'BLEU': 0.0,
                        'ANLS': 0.0, 'METEOR': 0.0, 'Char_Accuracy': 0.0,
                        'ground_truth_length': 0, 'prediction_length': 0})
                    pbar.update(1)

                    if consecutive_errors >= MAX_CONSECUTIVE_ERRORS:
                        pbar.write(" Restarting browser...")
                        try: driver.quit()
                        except: pass
                        time.sleep(5)
                        driver = create_driver()
                        consecutive_errors = 0
                    continue
    finally:
        try: driver.quit()
        except: pass
        print(" Browser closed.")

    save_checkpoint(all_results, year_results, total_samples - 1, OUTPUT_DIR)
    print("\n All samples processed!")
    return all_results, year_results


print(" Benchmark functions defined.")

## 8. Display Results

In [ ]:
def display_results(all_results, year_results):
    """Display comprehensive results"""
    print("\n" + "="*80)
    print(" SUBASA OCR — TEST RESULTS")
    print("="*80)

    df = pd.DataFrame(all_results)

    print("\n" + "="*80)
    print(" OVERALL AVERAGE METRICS")
    print("="*80)
    for metric in ['CER', 'WER', 'BLEU', 'ANLS', 'METEOR', 'Char_Accuracy']:
        if metric in df.columns:
            avg = df[metric].mean()
            d, q = ("(Lower is better)", "Lower is better") if metric in ['CER','WER'] else ("(Higher is better)", "Higher is better")
            print(f"{metric:15} {d}: {avg:7.4f}  ({q})")
    print(f"\n{'Total Samples':15}: {len(df)}")

    print("\n" + "="*80)
    print(" YEAR-WISE AVERAGE METRICS")
    print("="*80)
    year_summary = []
    for year, group in sorted(df.groupby('year')):
        year_summary.append({
            'Year': year, 'Samples': len(group),
            'CER': f"{group['CER'].mean():.4f}",
            'WER': f"{group['WER'].mean():.4f}",
            'BLEU': f"{group['BLEU'].mean():.4f}",
            'ANLS': f"{group['ANLS'].mean():.4f}",
            'METEOR': f"{group['METEOR'].mean():.4f}",
            'Accuracy%': f"{group['Char_Accuracy'].mean():.2f}%"
        })
    year_df = pd.DataFrame(year_summary)
    print("\n" + year_df.to_string(index=False))

    display_cols = ['sample_idx', 'year', 'CER', 'WER', 'BLEU', 'ANLS', 'METEOR', 'Char_Accuracy']

    print("\n" + "="*80)
    print("INDIVIDUAL SAMPLE RESULTS (First 10)")
    print("="*80)
    print("\n" + df[display_cols].head(10).to_string(index=False))

    print("\n" + "="*80)
    print(" BEST 5 (by Char Accuracy)")
    print("="*80)
    print("\n" + df.nlargest(5, 'Char_Accuracy')[display_cols].to_string(index=False))

    print("\n" + "="*80)
    print(" WORST 5 (by Char Accuracy)")
    print("="*80)
    print("\n" + df.nsmallest(5, 'Char_Accuracy')[display_cols].to_string(index=False))

    df.to_csv(os.path.join(OUTPUT_DIR, "test_metrics.csv"), index=False, encoding='utf-8')
    year_df.to_csv(os.path.join(OUTPUT_DIR, "year_wise_metrics.csv"), index=False)
    print(f"\n Results saved to {OUTPUT_DIR}/")

    print("\n" + "="*80)
    print(" SUMMARY STATISTICS")
    print("="*80)
    print(f"\nSamples with >90% accuracy: {len(df[df['Char_Accuracy'] > 90])}")
    print(f"Samples with >80% accuracy: {len(df[df['Char_Accuracy'] > 80])}")
    print(f"Samples with <50% accuracy: {len(df[df['Char_Accuracy'] < 50])}")
    print(f"\nMedian Character Accuracy: {df['Char_Accuracy'].median():.2f}%")
    print(f"Std Dev Character Accuracy: {df['Char_Accuracy'].std():.2f}%")

    return df, year_df


print(" Display functions defined.")

In [ ]:
if __name__ == "__main__":
    print("\n" + "="*80)
    print(" STARTING SUBASA OCR BENCHMARK")
    print("="*80)

    try:
        all_results, year_results = test_entire_dataset(resume=True)
        results_df, year_df = display_results(all_results, year_results)

        print("\n" + "="*80)
        print(" BENCHMARK COMPLETED!")
        print("="*80)
    except KeyboardInterrupt:
        print("\n  Paused. Run again to resume.")

## 10. Download Results

In [ ]:
import shutil
from IPython.display import FileLink

if os.path.exists(OUTPUT_DIR):
    if os.path.exists('test_results.zip'):
        os.remove('test_results.zip')
    shutil.make_archive('test_results', 'zip', OUTPUT_DIR)
    print(f" Created test_results.zip ({os.path.getsize('test_results.zip') / (1024*1024):.2f} MB)")
    display(FileLink('test_results.zip'))
else:
    print(" No results yet. Run the benchmark first.")